## Install and import what is necessary to carry out the training

In [ ]:
pip install openai matplotlib wandb scikit-learn ratelimit

In [ ]:
import tiktoken
import numpy as np
import json
import openai
from openai.types.fine_tuning import SupervisedMethod, SupervisedHyperparameters
from openai import OpenAI
import wandb
import pandas as pd
import sklearn.metrics
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from ratelimit import limits, sleep_and_retry
import matplotlib.pyplot as plt

In [ ]:
pip install --upgrade tiktoken

## TOKENIZE

In [ ]:
#This code loads a emotion training JSONL file into a list and prints the total number of examples.
data_path = "training_Emocion.jsonl"

# Load the dataset
with open(data_path, 'r', encoding='utf-8') as f:
    dataset = [json.loads(line) for line in f if line.strip()]

# Initial dataset stats
print("Num examples:", len(dataset))

In [ ]:
#This code loads a emotion validation JSONL file into a list and prints the total number of examples.
data_path = "validation_Emocion.jsonl" 

# Load the dataset
with open(data_path, 'r', encoding='utf-8') as f:
    dataset = [json.loads(line) for line in f if line.strip()]

# Initial dataset stats
print("Num examples:", len(dataset))

In [ ]:
#This code loads a emotion test JSONL file into a list and prints the total number of examples.
data_path = "test_Emocion.jsonl" 

# Load the dataset
with open(data_path, 'r', encoding='utf-8') as f:
    dataset = [json.loads(line) for line in f if line.strip()]

# Initial dataset stats
print("Num examples:", len(dataset))

In [7]:
#This code sets the text encoding using the "o200k\_base" tokenizer for tokenizing text with the tiktoken library.
encoding=tiktoken.get_encoding("o200k_base")

In [ ]:
# These functions calculate token counts for message data using a tokenizer and print statistical distributions of given values.
# simplified from https://github.com/openai/openai-cookbook/blob/main/examples/How_to_count_tokens_with_tiktoken.ipynb
def num_tokens_from_messages(messages, tokens_per_message=3, tokens_per_name=1):
    num_tokens = 0
    for message in messages:
        num_tokens += tokens_per_message
        for key, value in message.items():
            num_tokens += len(encoding.encode(value))
            if key == "name":
                num_tokens += tokens_per_name
    num_tokens += 3
    return num_tokens

def num_assistant_tokens_from_messages(messages):
    num_tokens = 0
    for message in messages:
        if message["role"] == "assistant":
            num_tokens += len(encoding.encode(message["content"]))
    return num_tokens

def print_distribution(values, name):
    print(f"\n#### Distribution of {name}:")
    print(f"min / max: {min(values)}, {max(values)}")
    print(f"mean / median: {np.mean(values)}, {np.median(values)}")
    print(f"p5 / p95: {np.quantile(values, 0.1)}, {np.quantile(values, 0.9)}")

In [ ]:
#This code analyzes the dataset for missing roles, 
#calculates token counts per message and conversation, and reports examples that exceed the  32,768 token limit.
#Statistical check. This will determine whether the data will be truncated, resulting in the loss of relevant information and the 
#model being unsuitable.
n_missing_system = 0
n_missing_user = 0
n_messages = []
convo_lens = []
assistant_message_lens = []

for ex in dataset:
    messages = ex["messages"]
    if not any(message["role"] == "system" for message in messages):
        n_missing_system += 1
    if not any(message["role"] == "user" for message in messages):
        n_missing_user += 1
    n_messages.append(len(messages))
    convo_lens.append(num_tokens_from_messages(messages))
    assistant_message_lens.append(num_assistant_tokens_from_messages(messages))
    
print("Num examples missing system message:", n_missing_system)
print("Num examples missing user message:", n_missing_user)
print_distribution(n_messages, "num_messages_per_example")
print_distribution(convo_lens, "num_total_tokens_per_example")
print_distribution(assistant_message_lens, "num_assistant_tokens_per_example")
n_too_long = sum(l > 32768   for l in convo_lens)  # Limit value of the model, GPT4.1-mini=  32.768 max token output
print(f"\n{n_too_long} examples may be over the 32.768   token limit, they will be truncated during fine-tuning")

In [ ]:
#This code calculates the optimal number of training epochs based on dataset size and token limits, 
#and estimates total token usage for billing during model training.
MAX_TOKENS_PER_EXAMPLE = 32768   # Limit value of the model, GPT4.1-mini= 32768 max token output
MIN_TARGET_EXAMPLES = 100
MAX_TARGET_EXAMPLES = 25000
TARGET_EPOCHS = 3   #epochs estimated 3
MIN_EPOCHS = 1
MAX_EPOCHS = 25

n_epochs = TARGET_EPOCHS
n_train_examples = len(dataset)
if n_train_examples * TARGET_EPOCHS < MIN_TARGET_EXAMPLES:
     n_epochs = min(MAX_EPOCHS, MIN_TARGET_EXAMPLES // n_train_examples)
elif n_train_examples * TARGET_EPOCHS > MAX_TARGET_EXAMPLES:
    n_epochs = max(MIN_EPOCHS, MAX_TARGET_EXAMPLES // n_train_examples)

n_billing_tokens_in_dataset = sum(min(MAX_TOKENS_PER_EXAMPLE, length) for length in convo_lens)
print(f"Dataset has ~{n_billing_tokens_in_dataset} tokens that will be charged for during training")
print(f"By default, you'll train for {n_epochs} epochs on this dataset")
print(f"By default, you'll be charged for ~{n_epochs * n_billing_tokens_in_dataset} tokens")


# Fine-tuning process execution

In [ ]:
#This code sets OpenAI API keys, initializes a Weights & Biases project for tracking, and creates an OpenAI client instance.
#API key for GCO account
openai.api_key= # <----------- Add your key

#Start wandb
#will guide you to add your API key in order to login 
wandb.init(project="Emotion analysis 4.1-mini") 

#Definition of the client
client = OpenAI(api_key= ) # <----------- Add your key

In [ ]:
# This code uploads training and validation JSONL files to OpenAI for fine-tuning and prints their assigned file IDs.
training_file = client.files.create(
  file=open("training_Emocion.jsonl", "rb"),
  purpose='fine-tune'
)
validate_file = client.files.create(
  file=open("validation_Emocion.jsonl", "rb"),
  purpose="fine-tune"
)

training_file_id = training_file.id
validate_file_id = validate_file.id
print("Object of training_emotion ", training_file)
print("Training_File has been uploaded to OpenAI with id ", training_file_id)
print("Validate_File has been uploaded to OpenAI with id ", validate_file_id)

In [ ]:
#This code retrieves all available models from the OpenAI client and filters the list to show only those with "gpt-4.1" in their model ID.
models=client.models.list()
models

gpt4_1_models = [m for m in models if "gpt-4.1" in m.id]
gpt4_1_models

In [ ]:
training_file_id
validate_file_id

In [ ]:
#This code initiates a fine-tuning job on the "gpt-4.1-mini" model 
#using uploaded training and validation files, sets hyperparameters, 
#integrates with Weights & Biases, and prints job details and ID.
ft_job = client.fine_tuning.jobs.create(
    training_file=training_file_id,
    validation_file=validate_file_id,
    model="gpt-4.1-mini-2025-04-14", 
    hyperparameters={
        "n_epochs": 3
    },
    integrations=[
        {
            "type": "wandb",
            "wandb": {
                "project": wandb.run.project,
                "name": "Emotion-4.1-mini" 
            }
        }
    ]
)

model_id=ft_job.fine_tuned_model
print("Object ", ft_job)
print("Fine Tune Job has been created with id ", ft_job.id)

# Fine-tuned model verification

In [ ]:
#This function sends a text to the fine-tuned GPT-4.1-mini model to get its emotion prediction, respecting rate limits to avoid API throttling.
RATE_LIMIT_TPM=200000 
@sleep_and_retry
@limits(calls=RATE_LIMIT_TPM, period=60) 
def  make_request(text):
    completion = client.chat.completions.create(
        model=model_idi,
        messages=[
            {"role": "system", "content": "What is the emotion of the following text? Respond with 'Anger', 'Fear', 'Happy', 'Sad', 'Neutral', 'Surprised' or 'Empathy'."},
            {"role": "user", "content": text},
        ]
    )
    return completion.choices[0].message.content

In [ ]:
#This code reads test data from Excel, sends texts to the model for emotions predictions, stores results with true labels, 
#and logs them to Weights & Biases for analysis.
filename = 'test_Emocion.xlsx'
df= pd.read_excel(filename)

model_idi= # <----------- Add your job ID

class_emotion=["Anger", "Fear", "Happy", "Sad", "Neutral", "Surprised", "Empathy"]

predictions=[] 
true_labels=[]

df_results = pd.DataFrame(columns=["Comment", "Prediction", "True label"])

for index, row in df.iterrows(): 
    text = row['MESSAGE']
    try: 
        
        response=  make_request(text)
        complete_predictions=response
        
        label=class_emotion.index(response)
        
        predictions.append(class_emotion[label])
        
        true_label = row['EMOTION']
        
        true_labels.append(true_label)
        df_results = pd.concat([df_results, pd.DataFrame({"Commnent": [text], "Prediction": [response], "True Label": [true_label]})], ignore_index=True)
        
    except Exception as e:
        print("Error:", e)
        continue
        
wandb.log({"predictions_vs_true_label_emotion-4.1-mini": wandb.Table(dataframe=df_results)}) 
        

### Classification report

In [ ]:
#This code generates and prints a classification report comparing the model's predicted emotions to the true labels, 
#including precision, recall, and F1-score metrics.
report = classification_report( true_labels, predictions)
print("Emotion Clasification Report (GPT4.1-mini):")
print(report)

In [ ]:
#Parses the `classification_report` output into a structured table and logs it to Weights & Biases for easy performance visualization.
lines = report.split('\n')

lines = [line for line in lines if line.strip()]

table_data = []

for line in lines:
    parts = line.split()
    parts = [part for part in parts if part]

    if len(parts) == 5 and parts[0] != 'accuracy':
        class_name = parts[0]
        precision = parts[1]
        recall = parts[2]
        f1_score = parts[3]
        support = parts[4]
        
        table_data.append([class_name, precision, recall, f1_score, support])


columns = ["Class", "Precision", "Recall", "F1-score", "Support"]

wandb.log({"classification_report_emotion-4.1-mini": wandb.Table(data=table_data, columns=columns)})

### Confusion Matrix

In [ ]:
#Generates and visualizes the confusion matrix of model predictions versus true labels, 
#saves it as an image, and logs it to Weights & Biases for tracking and analysis.
#Confusion matrix
matrix= confusion_matrix( true_labels, predictions, labels=class_emotion)

# Confusion matrix printing
print("Emotion confusion matrix (GPT4.1-mini):")
print(matrix)

vis=ConfusionMatrixDisplay(matrix, display_labels=class_emotion)
fig, ax = plt.subplots(figsize=(10, 8))
vis.plot(ax=ax, cmap=plt.cm.Blues) 

# Adjust the spacing between names on the x-axis
ax.set_xticklabels(class_emotion, rotation=45, ha="right")

plt.title("Emotion confusion matrix (GPT4.1-mini)")
plt.xlabel("Predicion")
plt.ylabel("True Labels")
plt.tight_layout()  #Adjust the layout to avoid overlaps

# Save the figure as an image file
img_path = "confusion_matrix-emotion-4.1-mini.png"
plt.savefig(img_path)

# Upload the image to wandb as an artifact
wandb.log({"confusion_matrix_emotion-4.1-mini": wandb.Image(img_path)})

plt.show()

In [ ]:
wandb.finish()